In [27]:
import os
import gc
import joblib
import numpy as np
import pandas as pd

from collections import defaultdict

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, SimpleRNN, LSTM, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [29]:
# =========================================================
# 0. 기본 설정
# =========================================================
path_2017 = r"D:\dataset\2017.csv"
path_2018 = r"D:\dataset\2018.csv"

chunk_size = 100_000
random_state = 42
target_per_class_train = 50_000   # 2018 train용 클래스별 최대 수
save_dir = r"D:\dataset\benchmark_reverse_2018train_2017test"
os.makedirs(save_dir, exist_ok=True)

np.random.seed(random_state)
tf.random.set_seed(random_state)

In [31]:
# =========================================================
# 1. 2018 -> 2017 컬럼명 매핑
# =========================================================
rename_2018_to_2017 = {
    "Dst Port": "Destination Port",
    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",
    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",
    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Bwd Pkt Len Max": "Bwd Packet Length Max",
    "Bwd Pkt Len Min": "Bwd Packet Length Min",
    "Bwd Pkt Len Mean": "Bwd Packet Length Mean",
    "Bwd Pkt Len Std": "Bwd Packet Length Std",
    "Flow Byts/s": "Flow Bytes/s",
    "Flow Pkts/s": "Flow Packets/s",
    "Fwd IAT Tot": "Fwd IAT Total",
    "Bwd IAT Tot": "Bwd IAT Total",
    "Fwd Header Len": "Fwd Header Length",
    "Bwd Header Len": "Bwd Header Length",
    "Fwd Pkts/s": "Fwd Packets/s",
    "Bwd Pkts/s": "Bwd Packets/s",
    "Pkt Len Min": "Min Packet Length",
    "Pkt Len Max": "Max Packet Length",
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Var": "Packet Length Variance",
    "FIN Flag Cnt": "FIN Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "ECE Flag Cnt": "ECE Flag Count",
    "Pkt Size Avg": "Average Packet Size",
    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
    "Bwd Seg Size Avg": "Avg Bwd Segment Size",
    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",
    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",
}

drop_cols = ["Protocol", "Timestamp", "Fwd Header Length.1"]

In [33]:
# =========================================================
# 2. 라벨 정리
# =========================================================
def clean_label(x):
    x = str(x).strip()
    x = x.replace("�", "-")
    x = x.replace("–", "-")
    x = x.replace("Web Attack � Brute Force", "Web Attack - Brute Force")
    x = x.replace("Web Attack � XSS", "Web Attack - XSS")
    x = x.replace("Web Attack � Sql Injection", "Web Attack - Sql Injection")
    if x.lower() == "benign":
        return "BENIGN"
    return x

def unify_label_reverse(label):
    """
    이번 실험 기준:
    Train=2018 라벨 기준으로 통일
    2017 쪽 동등 라벨은 2018 명칭으로 매핑
    """
    label = str(label).strip()
    label = label.replace("�", "-")
    label = label.replace("–", "-")

    if label.lower() == "benign":
        return "BENIGN"

    # Brute Force 계열: FTP / SSH는 분리 유지
    elif label in ["FTP-Patator", "FTP-BruteForce"]:
        return "FTP-BruteForce"

    elif label in ["SSH-Patator", "SSH-Bruteforce"]:
        return "SSH-Bruteforce"

    # Web Attack은 하나로 통합
    elif label in [
        "Web Attack - Brute Force",
        "Web Attack - XSS",
        "Web Attack - Sql Injection",
        "Web Attack - SQL Injection",
        "Web Attack- Brute Force",
        "Web Attack- XSS",
        "Web Attack- Sql Injection"
    ]:
        return "Web Attack"

    # DoS 계열 이름을 2018 쪽 표기 기준으로 통일
    elif label in ["DoS GoldenEye", "DoS attacks-GoldenEye"]:
        return "DoS GoldenEye"

    elif label in ["DoS Hulk", "DoS attacks-Hulk"]:
        return "DoS Hulk"

    elif label in ["DoS slowloris", "DoS attacks-Slowloris"]:
        return "DoS Slowloris"

    elif label in ["DoS Slowhttptest", "DoS attacks-SlowHTTPTest"]:
        return "DoS SlowHTTPTest"

    # DDoS / HOIC는 분리 유지
    elif label == "DDoS":
        return "DDoS"

    elif label == "DDOS attack-HOIC":
        return "DDOS attack-HOIC"

    else:
        return label


In [35]:
# =========================================================
# 3. 컬럼 확인 -> 공통 특성 계산
# =========================================================
head_2017 = pd.read_csv(path_2017, nrows=5)
head_2018 = pd.read_csv(path_2018, nrows=5)

head_2017.columns = head_2017.columns.str.strip()
head_2018.columns = head_2018.columns.str.strip()
head_2018.rename(columns=rename_2018_to_2017, inplace=True)

head_2017.drop(columns=[c for c in drop_cols if c in head_2017.columns], inplace=True)
head_2018.drop(columns=[c for c in drop_cols if c in head_2018.columns], inplace=True)

common_cols = list(set(head_2017.columns).intersection(set(head_2018.columns)))
common_cols = [c for c in common_cols if c != "Label"] + ["Label"]

print("공통 컬럼 수:", len(common_cols))
print("공통 컬럼 일부:", common_cols[:10])

공통 컬럼 수: 70
공통 컬럼 일부: ['Packet Length Mean', 'Bwd URG Flags', 'SYN Flag Count', 'Min Packet Length', 'Total Length of Fwd Packets', 'Idle Std', 'Fwd IAT Total', 'Bwd IAT Max', 'Total Backward Packets', 'Fwd URG Flags']


In [37]:
# =========================================================
# 4. chunk 전처리 함수
# =========================================================
def preprocess_chunk(df, is_2018=False):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if is_2018:
        df.rename(columns=rename_2018_to_2017, inplace=True)

    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    if "Label" not in df.columns:
        return None

    df["Label"] = df["Label"].apply(clean_label).apply(unify_label_reverse)

    missing_cols = [c for c in common_cols if c not in df.columns]
    for c in missing_cols:
        df[c] = np.nan

    df = df[common_cols]

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    if "Flow Duration" in df.columns:
        df = df[df["Flow Duration"] >= 0]

    df.dropna(subset=["Label"], inplace=True)

    feature_cols = [c for c in df.columns if c != "Label"]
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [39]:
# =========================================================
# 5. reservoir sampling (클래스별)
# =========================================================
def reservoir_sample_df(reservoir, chunk_df, target_n, seen_count, seed=42):
    rng = np.random.default_rng(seed + seen_count)

    if len(reservoir) < target_n:
        need = target_n - len(reservoir)
        take = min(need, len(chunk_df))

        if take > 0:
            if len(reservoir) == 0:
                reservoir = chunk_df.iloc[:take].copy()
            else:
                reservoir = pd.concat([reservoir, chunk_df.iloc[:take]], ignore_index=True)

        start_idx = take
    else:
        start_idx = 0

    for i in range(start_idx, len(chunk_df)):
        seen_count += 1
        j = rng.integers(0, seen_count + 1)
        if j < target_n:
            reservoir.iloc[j] = chunk_df.iloc[i].values

    return reservoir, seen_count

In [43]:
# =========================================================
# 6. 2018 train 수집 (클래스별 최대 5만)
# =========================================================
from collections import defaultdict
import pandas as pd

train_parts = defaultdict(list)
current_counts = defaultdict(int)

for chunk in pd.read_csv(path_2018, chunksize=chunk_size, low_memory=True):
    chunk = preprocess_chunk(chunk, is_2018=True)
    if chunk is None or len(chunk) == 0:
        continue

    for label, group in chunk.groupby("Label"):
        remain = target_per_class_train - current_counts[label]
        if remain <= 0:
            continue

        if len(group) > remain:
            group = group.sample(n=remain, random_state=random_state)

        train_parts[label].append(group)
        current_counts[label] += len(group)

train_df = pd.concat(
    [pd.concat(parts, ignore_index=True) for parts in train_parts.values()],
    ignore_index=True
)

print("\n[2018 Train] 클래스별 샘플 수")
print(train_df["Label"].value_counts().sort_values(ascending=False))


[2018 Train] 클래스별 샘플 수
Label
BENIGN              50000
Infilteration       50000
Bot                 50000
FTP-BruteForce      50000
SSH-Bruteforce      50000
DoS SlowHTTPTest    50000
DDOS attack-HOIC    44334
DoS GoldenEye       38532
DoS Hulk            36145
DoS Slowloris       10213
Name: count, dtype: int64


In [45]:
# =========================================================
# 7. 2017 test 수집 (전체 사용)
# =========================================================
test_chunks = []

for chunk in pd.read_csv(path_2017, chunksize=chunk_size, low_memory=True):
    chunk = preprocess_chunk(chunk, is_2018=False)
    if chunk is None or len(chunk) == 0:
        continue
    test_chunks.append(chunk)

test_df = pd.concat(test_chunks, ignore_index=True)
del test_chunks
gc.collect()

print("\n[2017 Test] 클래스 분포")
print(test_df["Label"].value_counts().sort_values(ascending=False))



[2017 Test] 클래스 분포
Label
BENIGN              201657
PortScan            100833
DoS Hulk            100833
DDoS                100833
DoS GoldenEye        10293
FTP-BruteForce        7938
SSH-Bruteforce        5897
DoS Slowloris         5796
DoS SlowHTTPTest      5499
Web Attack            2180
Bot                   1966
Name: count, dtype: int64


In [47]:
# =========================================================
# 8. train에 없는 test 클래스 제거
# =========================================================
train_classes = sorted(train_df["Label"].unique())
test_df = test_df[test_df["Label"].isin(train_classes)].copy()

print("\n학습 클래스:", train_classes)
print("\n필터 후 2017 Test 클래스 분포")
print(test_df["Label"].value_counts().sort_values(ascending=False))



학습 클래스: ['BENIGN', 'Bot', 'DDOS attack-HOIC', 'DoS GoldenEye', 'DoS Hulk', 'DoS SlowHTTPTest', 'DoS Slowloris', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce']

필터 후 2017 Test 클래스 분포
Label
BENIGN              201657
DoS Hulk            100833
DoS GoldenEye        10293
FTP-BruteForce        7938
SSH-Bruteforce        5897
DoS Slowloris         5796
DoS SlowHTTPTest      5499
Bot                   1966
Name: count, dtype: int64


In [49]:
# =========================================================
# 9. 결측치 처리
# =========================================================
feature_cols = [c for c in common_cols if c != "Label"]

for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)


In [51]:
# =========================================================
# 10. X / y 분리
# =========================================================
X_train = train_df.drop(columns=["Label"]).astype(np.float32)
y_train = train_df["Label"].copy()

X_test = test_df.drop(columns=["Label"]).astype(np.float32)
y_test = test_df["Label"].copy()

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)


X_train: (429224, 69)
X_test : (339879, 69)


In [53]:
# =========================================================
# 11. 라벨 인코딩
# =========================================================
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

num_classes = len(le.classes_)
print("\n클래스 순서:", list(le.classes_))

joblib.dump(le, os.path.join(save_dir, "label_encoder.pkl"))


클래스 순서: ['BENIGN', 'Bot', 'DDOS attack-HOIC', 'DoS GoldenEye', 'DoS Hulk', 'DoS SlowHTTPTest', 'DoS Slowloris', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce']


['D:\\dataset\\benchmark_reverse_2018train_2017test\\label_encoder.pkl']

In [55]:
# =========================================================
# 12. 스케일링
# =========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))

# RNN/LSTM/CNN 입력용 reshape
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_seq = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# DNN용 one-hot
y_train_cat = to_categorical(y_train_enc, num_classes=num_classes)
y_test_cat = to_categorical(y_test_enc, num_classes=num_classes)

In [57]:
# =========================================================
# 13. 공통 평가 함수
# =========================================================
def compute_security_metrics(y_true, y_pred, benign_label="BENIGN"):
    """
    FRR: 공격인데 BENIGN으로 예측된 비율
    FAR: BENIGN인데 공격으로 예측된 비율
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    attack_mask = (y_true != benign_label)
    benign_mask = (y_true == benign_label)

    if attack_mask.sum() > 0:
        frr = ((y_pred[attack_mask] == benign_label).sum()) / attack_mask.sum()
    else:
        frr = np.nan

    if benign_mask.sum() > 0:
        far = ((y_pred[benign_mask] != benign_label).sum()) / benign_mask.sum()
    else:
        far = np.nan

    return frr, far

def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    frr, far = compute_security_metrics(y_true, y_pred, benign_label="BENIGN")

    print(f"\n================ {model_name} ================")
    print("Accuracy:", acc)
    print("FRR     :", frr)
    print("FAR     :", far)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "Model": model_name,
        "Accuracy": acc,
        "FRR": frr,
        "FAR": far
    }

results = []

In [59]:
# =========================================================
# 14. RandomForest
# =========================================================
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=random_state,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(evaluate_model("RandomForest", y_test, y_pred_rf))

joblib.dump(rf, os.path.join(save_dir, "rf.pkl"))


================ RandomForest ================
Accuracy: 0.3446020495529291
FRR     : 0.49466076312019797
FAR     : 0.5986353064857655

Classification Report:
                  precision    recall  f1-score   support

          BENIGN       0.54      0.40      0.46    201657
             Bot       0.93      0.26      0.40      1966
DDOS attack-HOIC       0.00      0.00      0.00         0
   DoS GoldenEye       0.80      0.30      0.43     10293
        DoS Hulk       0.93      0.25      0.40    100833
DoS SlowHTTPTest       0.00      0.00      0.00      5499
   DoS Slowloris       0.49      0.71      0.58      5796
  FTP-BruteForce       0.00      0.00      0.00      7938
   Infilteration       0.00      0.00      0.00         0
  SSH-Bruteforce       0.47      0.48      0.47      5897

        accuracy                           0.34    339879
       macro avg       0.42      0.24      0.28    339879
    weighted avg       0.64      0.34      0.43    339879



['D:\\dataset\\benchmark_reverse_2018train_2017test\\rf.pkl']

In [60]:
# =========================================================
# 15. XGBoost
# =========================================================
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=num_classes,
    eval_metric="mlogloss",
    random_state=random_state,
    n_jobs=-1
)

xgb.fit(X_train, y_train_enc)
y_pred_xgb_enc = xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)
results.append(evaluate_model("XGBoost", y_test, y_pred_xgb))

joblib.dump(xgb, os.path.join(save_dir, "xgb.pkl"))


================ XGBoost ================
Accuracy: 0.5495043824419867
FRR     : 0.47320976400283604
FAR     : 0.27026088853845887

Classification Report:
                  precision    recall  f1-score   support

          BENIGN       0.69      0.73      0.71    201657
             Bot       0.95      0.62      0.75      1966
DDOS attack-HOIC       0.00      0.00      0.00         0
   DoS GoldenEye       0.75      0.89      0.81     10293
        DoS Hulk       0.95      0.20      0.33    100833
DoS SlowHTTPTest       0.00      0.00      0.00      5499
   DoS Slowloris       0.42      0.99      0.59      5796
  FTP-BruteForce       0.83      0.05      0.10      7938
   Infilteration       0.00      0.00      0.00         0
  SSH-Bruteforce       0.76      0.49      0.60      5897

        accuracy                           0.55    339879
       macro avg       0.54      0.40      0.39    339879
    weighted avg       0.76      0.55      0.57    339879



['D:\\dataset\\benchmark_reverse_2018train_2017test\\xgb.pkl']

In [61]:
# =========================================================
# 16. DNN
# =========================================================
dnn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

dnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

dnn.fit(
    X_train_scaled, y_train_cat,
    validation_split=0.2,
    epochs=30,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

y_pred_dnn_prob = dnn.predict(X_test_scaled, batch_size=2048, verbose=0)
y_pred_dnn = le.inverse_transform(np.argmax(y_pred_dnn_prob, axis=1))
results.append(evaluate_model("DNN", y_test, y_pred_dnn))

dnn.save(os.path.join(save_dir, "dnn.keras"))

Epoch 1/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7401 - loss: 0.6165 - val_accuracy: 0.1545 - val_loss: 6.6380
Epoch 2/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8233 - loss: 0.3164 - val_accuracy: 0.1545 - val_loss: 8.2381
Epoch 3/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8264 - loss: 0.3034 - val_accuracy: 0.1545 - val_loss: 8.7733
Epoch 4/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8274 - loss: 0.2957 - val_accuracy: 0.1389 - val_loss: 10.2131
Epoch 5/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8281 - loss: 0.2929 - val_accuracy: 0.1545 - val_loss: 11.7543
Epoch 6/30
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8297 - loss: 0.2906 - val_accuracy: 0.1545 - val_loss: 11.3085

================ DNN ================
Accuracy: 0.21221081620223667
FRR     : 0.32097640028360175
FAR     : 0.7424537705113138

Classification Report:
                  precision    recall  f1-score   support

          BENIGN  

In [62]:
# =========================================================
# 17. RNN
# =========================================================
rnn = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

rnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

rnn.fit(
    X_train_seq, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

y_pred_rnn_prob = rnn.predict(X_test_seq, batch_size=2048, verbose=0)
y_pred_rnn = le.inverse_transform(np.argmax(y_pred_rnn_prob, axis=1))
results.append(evaluate_model("RNN", y_test, y_pred_rnn))

rnn.save(os.path.join(save_dir, "rnn.keras"))


Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.7057 - loss: 0.7753 - val_accuracy: 0.1608 - val_loss: 6.2010
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8198 - loss: 0.3305 - val_accuracy: 0.1545 - val_loss: 7.1241
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.7867 - loss: 0.4433 - val_accuracy: 0.1628 - val_loss: 6.5703
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8140 - loss: 0.3474 - val_accuracy: 0.1598 - val_loss: 7.6612
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8220 - loss: 0.3168 - val_accuracy: 0.1587 - val_loss: 8.7451
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8221 - loss: 0.3132 - val_accuracy: 0.1595 - val_loss: 8.8383

================ RNN ================
Accuracy: 0.274524168895401
FRR     : 0.08355399285207854
FAR     : 0.6457450026530197

Classification Report:
                  precision    recall  f1-score   support

          B

In [63]:
# =========================================================
# 18. LSTM
# =========================================================
lstm = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

lstm.fit(
    X_train_seq, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

y_pred_lstm_prob = lstm.predict(X_test_seq, batch_size=2048, verbose=0)
y_pred_lstm = le.inverse_transform(np.argmax(y_pred_lstm_prob, axis=1))
results.append(evaluate_model("LSTM", y_test, y_pred_lstm))

lstm.save(os.path.join(save_dir, "lstm.keras"))


Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 67s 98ms/step - accuracy: 0.4696 - loss: 1.3594 - val_accuracy: 0.2033 - val_loss: 6.2153
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 65s 97ms/step - accuracy: 0.7749 - loss: 0.4773 - val_accuracy: 0.2033 - val_loss: 7.0747
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 62s 93ms/step - accuracy: 0.7706 - loss: 0.4868 - val_accuracy: 0.1662 - val_loss: 8.9283
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 62s 93ms/step - accuracy: 0.7730 - loss: 0.5010 - val_accuracy: 0.1673 - val_loss: 9.5104
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 64s 96ms/step - accuracy: 0.7970 - loss: 0.3972 - val_accuracy: 0.0245 - val_loss: 9.6572
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 65s 97ms/step - accuracy: 0.8074 - loss: 0.3617 - val_accuracy: 0.0337 - val_loss: 9.5262

================ LSTM ================
Accuracy: 0.15877415197761557
FRR     : 0.10899133278349322
FAR     : 0.8916576166460871

Classification Report:
                  precision    recall  f1-score   support

        

In [64]:
# =========================================================
# 19. CNN (1D)
# =========================================================
cnn = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),
    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

cnn.fit(
    X_train_seq, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

y_pred_cnn_prob = cnn.predict(X_test_seq, batch_size=2048, verbose=0)
y_pred_cnn = le.inverse_transform(np.argmax(y_pred_cnn_prob, axis=1))
results.append(evaluate_model("CNN", y_test, y_pred_cnn))

cnn.save(os.path.join(save_dir, "cnn.keras"))

Epoch 1/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - accuracy: 0.7520 - loss: 0.5812 - val_accuracy: 0.1595 - val_loss: 7.5461
Epoch 2/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.8264 - loss: 0.3076 - val_accuracy: 0.1587 - val_loss: 6.7717
Epoch 3/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.8290 - loss: 0.2939 - val_accuracy: 0.1587 - val_loss: 7.0699
Epoch 4/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - accuracy: 0.8306 - loss: 0.2894 - val_accuracy: 0.1628 - val_loss: 7.1203
Epoch 5/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.8308 - loss: 0.2878 - val_accuracy: 0.1660 - val_loss: 7.7363
Epoch 6/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.8316 - loss: 0.2872 - val_accuracy: 0.1622 - val_loss: 7.2919
Epoch 7/20
671/671 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.8320 - loss: 0.2859 - val_accuracy: 0.1660 - val_loss: 8.0675

================ CNN ================
Accuracy: 0.20281629638783213
FRR     : 0.086614287

In [65]:
# =========================================================
# 20. 결과 저장
# =========================================================
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
print("\n================ 최종 결과 요약 ================")
print(results_df)

results_df.to_csv(os.path.join(save_dir, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

# confusion matrix 저장 예시 (RF)
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=le.classes_)
cm_rf_df = pd.DataFrame(cm_rf, index=le.classes_, columns=le.classes_)
cm_rf_df.to_csv(os.path.join(save_dir, "rf_confusion_matrix.csv"), encoding="utf-8-sig")

print(f"\n저장 완료: {save_dir}")


================ 최종 결과 요약 ================
          Model  Accuracy       FRR       FAR
1       XGBoost  0.549504  0.473210  0.270261
0  RandomForest  0.344602  0.494661  0.598635
3           RNN  0.274524  0.083554  0.645745
2           DNN  0.212211  0.320976  0.742454
5           CNN  0.202816  0.086614  0.829056
4          LSTM  0.158774  0.108991  0.891658

저장 완료: D:\dataset\benchmark_reverse_2018train_2017test
